# Step 4 — Risk Classification Model
ThreadSense AI | JAK Threads

Trains a Decision Tree and Random Forest to predict `Slow_Moving_Risk` from the 9 verified engineered features. Run cells top to bottom.

## Step 1: Setup

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import joblib
import os

df = pd.read_excel('../data/processed/jak_threads_locked_in_verified.xlsx')
print(df.shape)
df.head()

## Step 2: Define X and y

In [ ]:
feature_cols = [
    'Inventory_Age_Days', 'Days_Since_Last_Sale', 'Sales_Velocity_7D',
    'Sales_Velocity_30D', 'Sell_Through_Rate', 'Days_Of_Inventory',
    'Sales_Acceleration', 'Gross_Margin_Pct', 'Stock_to_Sales_Ratio'
]

X = df[feature_cols]
y = df['Slow_Moving_Risk']

print(y.value_counts())

## Step 3: Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## Step 4: Train a Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

## Step 5: Evaluate on the test set

In [ ]:
y_pred_dt = dt_model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred_dt))
print('\nConfusion Matrix (rows=actual, cols=predicted):')
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_dt, labels=dt_model.classes_),
    index=dt_model.classes_, columns=dt_model.classes_
))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))

## Step 6: Feature importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

importance_df

## Step 7: Train and compare a Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('Decision Tree accuracy:', accuracy_score(y_test, y_pred_dt))
print('Random Forest accuracy:', accuracy_score(y_test, y_pred_rf))
print()
print(classification_report(y_test, y_pred_rf))

## Step 8: Save the winning model

In [ ]:
os.makedirs('../models', exist_ok=True)

# Swap to dt_model here if the Decision Tree wins on your run
joblib.dump(rf_model, '../models/risk_classifier.pkl')
print('Model saved to models/risk_classifier.pkl')